# 1.0 Load packages and data

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.impute import SimpleImputer

In [ ]:
# Load data
df = pd.read_csv("eliddi_episode_long.txt", sep="\t", encoding="utf-8", low_memory=False) # remove objects in data
codes = pd.read_csv("codes.csv") # load the data

# 2.0 Data previewing

## 2.1 TUS

In [ ]:
# view tus
df.head()

In [ ]:
# view data shape
df.shape

In [ ]:
# check columns
df.columns

In [ ]:
# check the data makeup
df.describe()

In [ ]:
# check the unique columns
df.nunique()

In [ ]:
# check the code columns
print("Number of unique codes in the primary column:", df["pri"].nunique())
print(df["pri"].unique())
print("-" * 70)
print("Number of unique codes in the secondary column:", df["sec"].nunique())
print(df["sec"].unique())

The "pri" (primary) and "sec" (secondary) columns hold the codes of activity in the survey dataset. These two have the same number of unique codes; the "pri" is the first activity that a respondent would call work, while the secondary is an addition. So the primary would be the one to merge the two datasets, because its time controls what is recorded for the secondary activity.

Not all the codes on the primary and secondary codes from the TUS data are described in the documentation. Notice that the "code" column from the codes table has 108 activity description columns while the "pri" and "sec" code columns from df are 120 apiece.

In [ ]:
# check the order of the diary
df.diaryord.unique()

In [ ]:
# check episode numbers
df.epnum.unique()

In [ ]:
# check unique modes
df['mode'].unique()

In [ ]:
# check unique modes
df['ontrip'].unique()

In [ ]:
# check the locations
df['loc'].unique()

## 2.2 Codes

In [ ]:
# view codes
codes.head()

In [ ]:
# check destribution
codes.describe()

In [ ]:
# check the unique counts
codes.nunique()

In [ ]:
# view the codes
codes.code.unique()

In [ ]:
# view activities and codes
codes[['code', 'activity']].head()

In [ ]:
# view the energy where used
codes['energy used'].unique()

In [ ]:
# view the energy where used
codes['where used'].unique()

# 3.0 Data cleaning

The columns with pri and sec are codes and contain similar codes on the codes tables. While the primary is the main actitivity, teh secondary is one done in addtion to the time the primary was ongoing. I will prioritise the primary column.

In [ ]:
# Rename the pri column with code for merging
df = df.rename(columns={"pri": "code"}, errors='raise')

In [ ]:
# Merge the tables on code columns using a one-to-one relationship
df_merged = df.merge(codes, on="code", how="left")
df_merged.head(3)

## 3.1 Sample selection

Our work is based off of Lorincz et al.(2021), which identified the appliances that led to energy consumption in homes. They highlighted activities such as dishwashing, ironing, food preparation, house cleaning, laundry, and TV watching as the major energy-using activities.

In [ ]:
# Select columns where energy is used in the home during weekdays 
df_elect = df_merged[(
    df_merged["energy used"] == "Yes") &        # choose the energy using activities
    (df_merged["loc"] == 1) &                  # location is in the house
    (~df_merged["dday"].isin([6,7]))]           # choose the weekdays

df_elect

In [ ]:
# Select columns needed for the analysis
selected_cols = ['mainid', 'diaryord', 'epnum', 'start', 'end', 'time', 'clockst', 'ww_alone',
                 'ww_partner', 'ww_child', 'ww_young', 'ww_otherhh', 'ww_othernonhh',
                 'ww_otherchild', 'dyear', 'dmonth', 'ddomonth', 'dday', 'sex', 'age',
                 'n_in_hh', 'nchild', 'nunder5', 'n5to11', 'n12to16', 'marstat', 'region',
                 'econstat', 'emplnow', 'tenure','hhincom2','headecon',
                 'wkhrnorm', 'activity', 'badcase']

# Convert the columns to a dataframe, df_elect
df_elect = df_elect[selected_cols]

In [ ]:
df_elect.head()

The unique households in the dataframe is now 1,799, reducing from 2174, since some households filled the survey during weekends and outdoors activities.

## 3.2 Standardise column

In [ ]:
df_elect = df_elect.rename(
    columns={
        'mainid':'mainid', # maninid: mainid;
        'diaryord': 'diary_order',  # diaryord: Diary order;'epnum': 'episode_number',
        'epnum': 'episode_number',
        'start': 'activity_startminute', # Start minute
        'end': 'activity_endminute', # End minute
        'time': 'duration_minutes',    # Duration of activity in minutes
        'clockst': 'starttime_24hr', # Start time on 24-hour clock
        'ww_alone': 'worked_alone',   # Worked alone
        'ww_partner': 'worked_with_partner',   # Worked with partner
        'ww_child': 'worked_with_under12',   # Worked with under12 child
        'ww_young': 'worked_with_age12to16',   # Worked with child with 12 to 16 years
        'ww_otherhh': 'worked_with_other',     # Worked with other household member
        'ww_othernonhh': 'worked_with_nonhhold_member',  # Worked with other non-household member
        'ww_otherchild': 'worked_with_nonhhold_child',     # Worked with other non-household child
        'dyear': 'diary_yr',       # dyear: Year in which diary was completed;
        'dmonth': 'diary_month',    # dmonth: Month in which diary was completed
        'ddomonth': 'day_of_month',  # Day of the month in which diary was completed: 11 - 19
        'dday': 'day_of_week',      # dday: Day of the week;
        'sex': 'sex',                 # sex: Sex of diarist;
        'age': 'age',   # age: Age of diarist;
        'n_in_hh': 'hh_size',   # n_in_hh: Household size;
        'nchild': "children_count",     # nchild: Number of children in household;
        'nunder5': 'under_age5',  # nunder5: Number of children under 5 years of age;
        'n5to11': 'age5to11',       # n5to11: Number of children aged 5 to 11;
        'n12to16': 'age12to16',         # n12to16: Number of children aged 12 to 16;
        'marstat': 'marital_status',    # marstat: Marital status
        'region': 'region',     # region: Region;
        'econstat': 'resp_occ_group',    # econstat: Main occupation;
        'emplnow': 'resp_empl',     # emplnow: Which of these best describes how you are working now?
        'tenure': 'housing_tenure',  # tenure: Housing tenure
        'hhincom2': 'hh_income',     # hhincom2: Annual household income from any source (after taxes/deductions);
        'headecon': 'chief_income_occ_group',    # Main occupation of the chief income earner
        'wkhrnorm': 'work_hours',  #    Hours worked in a normal week;
        'activity': 'activity_description', # activity: code description;
        'badcase': 'flags'  # low quality diary flag
    }, errors='raise')

Other columns descriptions are below.

Next operation is to replace values to ensure uniformity, identify missingness to maintain accuracy in the analysis. Changing the data types of the columns will ensure that subsequent operations are efficiently handled.

## 3.4 Fix the flags

0 Good quality diary        \
1 Sex or age is missing \
2 Day of the week is missing           \
3 90+ minutes missing \
4 Less than 7 episodes                 \
5 3+ basic activities are missing \
6 More than one condition missing

I will discretionarily keep households where age and sex are missing, because these columns were primarily about the diarist (respondent) and not the ages of the people in the household. 

In [ ]:
# View less than 7 episodes
df_flag4 = df_elect[df_elect['flags'].isin([4])]

Washing and dressing are the only activites where households provided less episodes. These two households spent 60 minutes  and 250 minutes respectively, on this activity, both with only two episodes.

In [ ]:
df_flag4['hh_size'].unique()

In [ ]:
# View the household characteristics for these subset
df_flag4[['mainid', 'hh_size','children_count', 'under_age5', 'age5to11', 'age12to16', 'duration_minutes', 'activity_description']]

I think household size 1 cannot reasonably do 250 minutes in 2 episodes of washing and dressing. I would remove that from the data. 60 Minutes for a household of 5 is reaseonable (my thoughts).

In [ ]:
# remove the rows where they are
df_elect = df_elect.drop(df_flag4.index, axis=0).copy()

Next is to remove where there are 3+ activities missing and conditions are not met. While these conditions are not specified in the codebook, I will go ahead to remove where these data are noted.

In [ ]:
# View flags 5 and 6
df_flag5n6 = df_elect[df_elect['flags'].isin([5, 6])]
df_flag5n6

In [ ]:
# Remove the rows from df_elect
df_elect = df_elect.drop(df_flag5n6.index, axis=0)

All the flags are addressed

## 3.5 Activity and region description

In [ ]:
# Check the activity descriptions
print('Activity descriptions:', df_elect['activity_description'].unique())

print('--' * 20) # demarcate the print with dash

# Check the region of households
print('Region:', df_elect['region'].unique())

I will merge 'Cleaning dwelling' and 'Cleaning garden, yard' to 'House Cleaning', and remove observations where gymnastic and listening to radio are present, as they will not represent energy consumption activities being considered.

In [ ]:
# Reduce the activities
# Unify the activities and Create a column for the new activity_description and remove Gymnastics
df_elect['activity_description'] = df_elect['activity_description'].replace(
    {'Washing and dressing': 'Washing & Dressing', 'Cleaning dwelling': 'House Cleaning', 'Watching TV, video, or DVD': 'Watching TV',
     'Watching/streaming TV/DVD, music': 'Watching TV', 'Food preparation and baking': 'Food Preparation', 'Ironing': 'Laundry',
     'Cleaning garden, yard': 'House Cleaning', 'Dish clearing, washing': 'Dishwashing', 'Heating dwelling and water': 'Heating House'}) 

# Remove gymnastics and listening to radio
df_elect = df_elect[~df_elect['activity_description'].isin(['Gymnastics and fitness', 'Listening to radio or recordings'])]
df_elect['activity_description'].unique()

In [ ]:
# Replace region column with readable entries
# 1 North East                   # 2 North West                      # 3 Yorkshire and The Humber
# 4 East Midlands                # 5 West Midlands                   # 6 East of England
# 7 London                       # 8 South East                      # 9 South West
# 10 Wales                       # 11 Scotland                       # 12 Northern Ireland
# 13 Channel Islands             # 14 Isle of Man

region = {1: "North East", 2: "North West", 3: "Yorkshire and The Humber",
          4: "East Midlands", 5: "West Midlands", 6: "East of England",
          7: "London", 8: "South East", 9: "South West", 10: "Wales",
          11: "Scotland", 12: "Northern Ireland", 13: "Channel Islands",
          14: "Isle of Man"}

# map to the regions
df_elect['region'] = df_elect['region'].astype('object').replace(region)
df_elect['region'].unique()

## 3.6 Date and time

In [ ]:
# Check the clock starttime
print('Starttime in 24 hour clock:', df_elect['starttime_24hr'].unique())
print("---" * 20)   # demarcate the prints

# Check the unique values of the time columns
print('Activity starting time in minutes:', df_elect['activity_startminute'].unique())
print("---" * 20)   # demarcate the prints
print('Activity ending time in minutes:', df_elect['activity_endminute'].unique())
print("---" * 20)  # demarcate the prints
print('Activity duration in minutes:', df_elect['duration_minutes'].unique())

The starttime_24hr column is in decimal; the period was supposed to show the column sign of a time format.

In [ ]:
# Compare times
df_elect[['starttime_24hr', 'activity_startminute', 'activity_endminute','duration_minutes']]

Clearly, 9 o'clock is not the 300th minute of the day. The survey start time was 4:00 am in the morning, which means there was an offset by four hours (240 minutes) on the activity start and end times. These columns will be corrected to start from zero minute, the normal midnight hour, by adding 240 minutes to the times, and will correspond to the starttime_24hr column.

In [ ]:
# Convert decimal time to proper time format
# Use a function; replace is changing the position of the zero
def decimal_to_time(decimal_time):
    hours = int(decimal_time)  # the whole number is the hour 
    minutes = round((decimal_time - hours) * 100)  # extract the fractional digits as minutes
    return f"{hours:02d}:{minutes:02d}"  # convert to the HH:MM format

# Apply function to the clock time
df_elect['starttime24_hhmm'] = df_elect['starttime_24hr'].apply(decimal_to_time)
df_elect['starttime24_hhmm'].unique()

In [ ]:
# Correct the start and end time for activity to correspond to clock time
# Add 240 minutes to the startminute column
df_elect["activity_startminute"] = (df_elect["activity_startminute"] + (4 * 60))  # add a 4 hour-minutes to the activity start time
df_elect["activity_endminute"]   = (df_elect["activity_endminute"]   + (4 * 60))  # add a 4 hour-minutes to the activity end time

In [ ]:
# Create activity endtime
# Convert the time string to datetime
df_elect['starttime24_hhmm'] = pd.to_datetime(df_elect['starttime24_hhmm'], format='%H:%M')

# Add the duration in minutes
df_elect['endtime24_hhmm'] = df_elect['starttime24_hhmm'] + pd.to_timedelta(df_elect['duration_minutes'], unit='m')

# Keep both as HH:MM time format (HH:MM)
df_elect['endtime24_hhmm'] = df_elect['endtime24_hhmm'].dt.strftime('%H:%M')  # convert back to string
df_elect['starttime24_hhmm'] = df_elect['starttime24_hhmm'].dt.strftime('%H:%M')

In [ ]:
# Create startdatetime for datetime[ns] datatype conversion by combining date and time
df_elect['startdatetime'] = (df_elect['diary_yr'].astype(str) + '-' +
                             df_elect['diary_month'].astype(str) + '-' +
                             df_elect['day_of_month'].astype(str))

# Include the time part
df_elect['startdatetime'] = pd.to_datetime(df_elect['startdatetime'] + 
                                           ' ' +          # Add space
                                           df_elect['starttime24_hhmm'])

# Create complementary enddatetime by adding duration to startdatetime
df_elect['enddatetime'] = df_elect['startdatetime'] + pd.to_timedelta(df_elect['duration_minutes'], unit='min')

This will give us the idea of when a person worked from friday into the weekend or from sunday of a weekend into monday.

In [ ]:
# Confirm mapping implemented for midnight crossover
df_elect_morethan_1440 = df_elect[df_elect['activity_endminute'] > 1439]

# Select only the time columns
df_elect_morethan_1440 = df_elect_morethan_1440[['activity_startminute', 'activity_endminute','duration_minutes', 'starttime_24hr',
                        'starttime24_hhmm', 'endtime24_hhmm', 'startdatetime', 'enddatetime']]

# Run this a couple of times to view the different samples of 
df_elect_morethan_1440.sample(5)

## 3.7 Socioeconomic features

Clean housing tenure

In [ ]:
# 1 Owns Outright        2 Owns on mortgage          3 Shared ownership                            4 Rents council
# 5 Rents Housing Association/charitable trust/new town development corporation
# 6 Rents Property company                           7 Rents Employer of a household member
# 8 Rents Other organisation                         9 Rents - Relative/friend (before living here) of a household member
# 10 Rents - Other individual/private landlord      11 Rent free, squatting                       12 Other (Please describe)

# Rename the tenure column and merge the tenure types
housing_tenure = {1: 'Owns Outright', 2: 'Owns on mortgage', 3: 'Shared ownership',
                  4: 'Rents council', 5: 'Rents Housing Association',
                  6: 'Rents Property company', 7: 'Rents Employer of a household member',
                  8: 'Rents Other organisation',   9: 'Rents Relative', 
                  10: 'Rents private landlord', 11: 'Rent free', 12: 'Other'}

# Apply to the tenure column
df_elect['housing_tenure'] = df_elect['housing_tenure'].replace(housing_tenure)
df_elect['housing_tenure'].unique()

Regroup the tenures

In [ ]:
# Regroup the tenure
tenure = {'Owns Outright': 'Owner', 'Owns on mortgage': 'Mortgage owned', 'Shared ownership': 'Shared ownership',
          'Rents private landlord': 'Rented from Private Landlord', 'Rents council': 'Rents Council',
          'Rent free': 'Rent free', 'Rents Housing Association': 'Third-party owned',
          'Rents Property company': 'Third-party owned', 'Rents Employer of a household member': 'Third-party owned',
          'Rents Other organisation': 'Third-party owned', 'Rents Relative': 'Third-party owned', 'Other': 'Other'}

# map the categories
df_elect['tenure'] = df_elect['housing_tenure'].replace(tenure)
df_elect['tenure'].unique()

Group the household sizes

In [ ]:
# Categorise household size
# All numbers are numeric. Only 7 is 'categorical' in the sense that it represents 7 and above.
size = {1 : 'small', 2 : 'small', 3 : 'medium', 4 : 'medium', 5 : 'medium', 6 : 'large', 7 : 'large'}

# Create column for the categories
df_elect['hhold_size_group'] = df_elect['hh_size'].astype('object').replace(size)
df_elect['hhold_size_group'].unique()

Group the Household annual income

In [ ]:
# 1 Under £10,000                   # 2 £10,000 to £19,999                   # 3 £20,000 to £29,999
# 4 £30,000 to £39,999              # 5 £40,000 to £49,999                   # 6 £50,000 to £59,999
# 7 £60,000 to £69,999              # 8 £70,000 to £79,999                   # 9 £80,000 to £89,999
# 10 £90,000 to £100,000            # 11 Over £100,000                       # 99 Don’t know/Prefer not to say

# Repace the integers with their corresponding range
df_elect['hh_income'] = df_elect['hh_income'].replace(
    {1: '0-9999', 2: '10000-19999', 3: '20000-29999',
     4: '30000-39999', 5: '40000-49999', 6: '50000-59999',
     7: '60000-69999', 8: '70000-79999', 9: '80000-89999',
     10: '90000-100000', 11: '>100000', 99: "Not Given"}) # 99 was not part of the responses recorded

# Confirm change is effected
df_elect['hh_income'].unique()

Applying a similar categorisation by Torriti and Yunusov helped the model classification and specificity.

Ref: https://www.sciencedirect.com/science/article/pii/S2214629620302723?via%3Dihub

In [ ]:
# Define income categories
low_income = ['0-9999', '10000-19999']
lower_middleincome = ['20000-29999']
middle_income = ['30000-39999']
upper_middleincome = ['40000-49999']
high_income = ['50000-59999']
very_highincome = ['60000-69999', '70000-79999', '80000-89999', '90000-100000', '>100000']

# List categories
categories = ['low income', 'lower middle income', 'middle income',
             'upper middle income', 'high income', 'very high income']

In [ ]:
# Set conditions
conditions = [df_elect['hh_income'].isin(low_income),  # if entry is in low-income
              df_elect['hh_income'].isin(lower_middleincome),    # if entry is in middle-income
              df_elect['hh_income'].isin(middle_income),         # if entry is in middle-income
              df_elect['hh_income'].isin(upper_middleincome),    # if entry is in upper-income 
              df_elect['hh_income'].isin(high_income),       # if entry is in high-income 
              df_elect['hh_income'].isin(very_highincome)]     # if entry is in very high-income 

In [ ]:
# Create income category column
df_elect['income_category'] = np.select(conditions, categories, default=df_elect['hh_income'])
df_elect['income_category'].unique()

Group the Household employment

In [ ]:
# 1 Self-employed with no employees            2 Self-employed with 1-25 employees
# 3 Employed      4 Partner/director/owner company of 25+  employees         5 Student
# 6 Casual worker – not in permanent employment     7 Housewife/ Homemaker
# 8 Retired and living on state pension          9 Retired and NOT living on state pension
# 10 Unemployed or not working due to lon-term sickness; less than 6 months
# 11 Unemployed or not working due to lon-term sickness; 6 months or more
# 12 Full-time carer of other household member   99 Prefer not to say

# Code the employment columns
occ_group = {1: 'Self-employed without employees', 2: 'Self-employed with 1-25 employees',
             3: 'Employed', 4: 'Partner/director/owner company of 25+ employees', 5: 'Student',
             6: 'Casual worker without permanent employment', 7: 'Housewife/ Homemaker',
             8: 'Retired and living on state pension', 9: 'Retired and not living on state pension',
             10: 'Unemployed or not working due to long-term sickness; less than 6 months',
             11: 'Unemployed or not working due to long-term sickness; 6 months or more',
             12: 'Full-time carer of other household member',  99: 'Prefer not to say'}

# Apply to the column
df_elect[["resp_occ_group", "chief_income_occ_group"]] = df_elect[
    ["resp_occ_group", "chief_income_occ_group"]].replace(occ_group)

In [ ]:
# View the changes
print("Respondent's work group:", df_elect["resp_occ_group"].unique())
print("---" * 20)  # demarcate the printouts
print("Chief income earner's work group:", df_elect["chief_income_occ_group"].unique())

There is no way of knowing how many people in the household are employed or not. We will be using the chief income earner's work group and the respondent's to make our assumptions. I will apply a higher weighting to 'employed' entries, followed by 'retired', then 'unemployed,' and 'missing.' Where any of employed and unemployed appear on either columns, the household is named partly employed but they are fully employed if both columns are employed. Same logiic is applied to retired and missing.

In [ ]:
# Group employement types
# Missing values
missing_values = [-8, 999, -9]

# Employment types
employed = ['Employed', 'Self-employed without employees', 'Self-employed with 1-25 employees',
            'Partner/director/owner company of 25+ employees', 'Casual worker without permanent employment']
 
# Unemployment types  
unemployed = ['Unemployed or not working due to long-term sickness; 6 months or more',
              'Housewife/ Homemaker', 'Student', 'Full-time carer of other household member',
              'Unemployed or not working due to long-term sickness; less than 6 months']

# Retirement types           
retired = ['Retired and living on state pension', 'Retired and not living on state pension']

In [ ]:
# Categorise household employment
def categorise_employment(row):
    resp = row['resp_occ_group']
    chief = row['chief_income_occ_group']
    
    # Both in same category
    if resp in employed and chief in employed:  # when both are employed
        return 'fully employed'
    if resp in retired and chief in retired:    # when both are retired
        return 'fully retired'
    if resp in unemployed and chief in unemployed:   # when both are unemployed
        return 'fully unemployed'  
    if resp in missing_values and chief in missing_values:      # when both are missing
        return 999
    
    # At least one employed (highest priority)
    if resp in employed or chief in employed:    # when one is employed and the other is either retired, unemployed, or missing
        return 'partly employed'
    
    # At least one retired (second priority)
    if resp in retired or chief in retired:    # when one is retired and the other is either unemployed or missing
        return 'partly retired'
    
    # At least one unemployed
    if resp in unemployed or chief in unemployed:  # when one is unemployed and the other is missing
        return 'partly unemployed'
    
    return 999        # if the other combinations of missing and not applicable, return missing

In [ ]:
# Apply the function
df_elect['household_employment_group'] = df_elect[['resp_occ_group', 'chief_income_occ_group']].apply(
    categorise_employment, axis=1)

# Confirm counts
df_elect['household_employment_group'].value_counts()

Group the respondents' work type

The survey did not allow for identifying if a person is at home. By using the respondent's work type, we can understand if some one is usually at home.

In [ ]:
# 1 working full time, travelling to my place of work on most days    
# 2 working part time, travelling to my place of work on most days
# 3 working full time, usually working from home       4 I am working part time, usually working from home
# 5 I have been furloughed by my employer              6 I'm not working – I am between jobs, unemployed
# 7 I'm not working – on parental leave                8 I'm not working – I am a homemaker
# 9 I'm not working – I am studying                    10 I'm not working – retired            98 Other

# Create respondents' work type as a pointer for availability
emplnow = {1: 'onsite full time', 2: 'onsite part time', 3: 'remote full time', 4: 'remote part time',
             5: 'furloughed', 6: 'unemployed', 7: 'parental leave', 8: 'homemaker', 9: 'studying',
             10: 'retired', 98: 'other'}

df_elect['resp_empl'] = df_elect['resp_empl'].replace(emplnow)
df_elect['resp_empl'].unique()

In [ ]:
# Group the respondents' type
work_type = {'remote part time': 'remote', 'remote full time': 'remote',
             'onsite full time': 'onsite', 'onsite part time': 'onsite',
             'retired': 'retired','other': 'other', 'furloughed': 'unemployed',
             'parental leave': 'parental leave', 'homemaker': 'homemaker',
             'studying': 'unemployed', 'unemployed': 'unemployed'}

# Apply to column
df_elect['resp_work_type'] = df_elect['resp_empl'].replace(work_type)
df_elect['resp_empl'].unique()

## 3.8 Missing Values

### 3.8.1 Identify Missing Values and Not Applicable Responses

The missing convention in the dataset are -8, -9, and 999.

In [ ]:
# Check the percentage of missing values
percent_missing = (((df_elect.isin([-8, -9, 999, '-8', '-9', '999'])).sum())/(len(df_elect))) * 100
percent_missing.sort_values(ascending=False)

A series of questions were asked where the responses were not applicable, represented by -7 and 99. These are not missing values but a set of derived questions that may not apply to the respondent. I will check their impact on the dataset.

In [ ]:
# The percentage of nonresponsive entries in the columns
nonresponsive_percent = ((df_elect.isin([-7, 98, 99]).sum())/(len(df_elect))) * 100
nonresponsive_percent.sort_values(ascending=False).head(20)

### 3.8.2 Fix datatypes

In [ ]:
# Select all categorical columns
categorical_cols = ['mainid', 'diary_order', 'sex', 'age',  'marital_status', 'region',
                    'resp_empl', 'housing_tenure', 'tenure', 'household_employment_group', 'resp_work_type',
                    'activity_description', 'income_category', 'endtime24_hhmm', 'starttime24_hhmm',
                   'worked_alone', 'worked_with_partner', 'worked_with_under12', 'hh_income',
                    'worked_with_age12to16', 'worked_with_other', 'worked_with_nonhhold_member',
                    'worked_with_nonhhold_child']

# Select all numerical columns
numerical_cols = ['children_count', 'episode_number', 'hh_size', 'under_age5', 'age5to11', 'age12to16','activity_startminute',
                  'activity_endminute', 'duration_minutes', 'day_of_week', 'work_hours']

# Select data columns
date_cols = ['startdatetime', 'enddatetime']

In [ ]:
# convert all the columns selected to categorical datatype
for col in categorical_cols:
    df_elect[col] = df_elect[col].astype("category")

# convert all the numeric columns selected to numeric datatype
for col in numerical_cols:
    df_elect[col] = pd.to_numeric(df_elect[col], errors='coerce')

# convert to datetime
for col in date_cols:
    df_elect[col] = pd.to_datetime(df_elect[col], errors='coerce')

### 3.3 Handle 'not applicable' and missing values

In [ ]:
# Drop unseful columns and with large missing values
df_elect = df_elect.drop(
    ['diary_yr', 'diary_month', 'age', 'day_of_month', 'hh_size',
     'resp_occ_group', 'chief_income_occ_group', 'housing_tenure',
     'hh_income', 'resp_empl', 'starttime_24hr', 'work_hours'], axis=1)

In [ ]:
# Separate columns
cat_cols = df_elect.select_dtypes(include=["object", 'category']).columns
num_cols = df_elect.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
# Address all not applicable conventions
# First convert categorical to string type
df_elect[cat_cols] = df_elect[cat_cols].astype(str)

# Replace -7 in categorical with 'not applicable', numerical with 0
df_elect[cat_cols] = df_elect[cat_cols].replace([-7, '-7'], 'not applicable')
df_elect[num_cols] = df_elect[num_cols].replace(-7, 0)

# Replace 'prefer not to say' in categorical with 'not given' and numerial with 0
df_elect[cat_cols] = df_elect[cat_cols].replace([99, '99'], 'not given')
df_elect[num_cols] = df_elect[num_cols].replace(99, 0)

# Replace other in categorical with other
df_elect[cat_cols] = df_elect[cat_cols].replace([98, '98'], 'other')
df_elect[num_cols] = df_elect[num_cols].replace(98, 0)

Income and tenure have higher missing values, above 10%. Dropping these will make the data lose alot of features. One can choose to drop all missing values, but that will take away the features that need to be understood. I will address the missing values with imputation, using median for numerical columns and mode for categorical.

In [ ]:
# Impute the columns appropriately
# Convert the missing conventions to not a number
df_elect = df_elect.replace([-8, -9, 999, '-8', '-9', '999'], np.nan)

# Impute categorical columns with mode
cat_imputer = SimpleImputer(strategy='most_frequent')
df_elect[cat_cols] = cat_imputer.fit_transform(df_elect[cat_cols])

# Impute the numerical columns with median
num_imputer = SimpleImputer(strategy='median')
df_elect[num_cols] = num_imputer.fit_transform(df_elect[num_cols])

You can choose to comment the chunk above, if the approach preferred is the removal of all missing values.

In [ ]:
# Confirm missing values and non-responsives are addressed
missing_summary = (((df_elect.isin(
    [-7, -8, -9, 99, 999, '-7', '-8', '-9', '99', '999',])   
                    ).sum())/(len(df_elect))) * 100

missing_summary.sort_values(ascending=False).head()

# 4.0 Data resampling

All the data has been cleaned, but I will resample the dataframe and create 30-minute bins that will allow for the analysis and understanding of what times of the day households are most actively engaged in any tracked activity at a high level. Since activities have start times, end times, and durations, I will analyse these activities in 30-minute time bins to see patterns throughout the day. This involve creating a fixed reference of all 48 time bins in a day (00:00-00:29, 00:30-00:59, ..., 23:30-23:59), match each activity to the bins it overlaps with and calculating the exact overlap duration for each activity-bin into a bin. An instance cooking recorded from 09:15 to 10:05 (50 minutes)
1. All bins predefined: ..., 09:00-09:30, 09:30-10:00, 10:00-10:30, ...
2. Check which bins the activity overlaps:
a. 09:00-09:29: overlaps from 09:15 to 09:29 = 15 minutes
b. 09:30-09:59: overlaps from 09:30 to 09:59 = 30 minutes
c. 10:00-10:29: overlaps from 10:00 to 10:05 = 5 minutes 
Total = 15 + 30 + 5 = 50 minutes (matches original duration)

Process
1. Create all 48 fixed half-hour bins for a 24-hour day (09:00-09:30, 09:30-10:00, 10:00-10:30 ..., 23:30-23:59) using the activity start time and end time columns.
2. Match each activity to all time bins it overlaps with and calculate the exact overlap duration for each activity-bin pair
3. Apply through across the dataset such that one row per activity time, overlap with time-bin overlap

## 4.1 Create 48-hour fixed half bins of a 24-hour day

In [ ]:
# Define a function to create time bins across a 24-hour day
def create_time_bins(interval=30):
    # Initialise an empty list to store the start and end minute of each bin
    bins = []

    # Loop through the full day in steps of the chosen interval
    for start in range(0, 1440, interval):
        # Define the end of the bin, making sure it does not go beyond 23:59
        end = min(start + interval - 1, 1439)  # e.g. 0-29, 30-59, ..., 1410-1439

        # Add the time bin as a tuple of (start minute, end minute)
        bins.append((start, end))

    # Return the full list of time bins
    return bins


# Create 30-minute time bins for the full day
time_bins = create_time_bins(30)

Create an xlookup-like function, which will be comparing the activity start and end times with the time bin start and end. The process will take about 2 minutes to run depending of laptop speed.

In [ ]:
# Define a function to expand activities into the time bins they overlap with
def create_activities_minutes(df, time_bins):
    # Initialise an empty list to store the expanded activity records
    expanded = []

    # Loop through each activity record to extract its start and end times
    for idx, row in df.iterrows():
        start = row["activity_startminute"]
        end = row["activity_endminute"]
        
        # Handle activities that cross midnight by extending the end time
        # into the next day
        if end < start:
            end += 1440
        
        # Check each time bin to see whether the activity overlaps with it
        for bin_start, bin_end in time_bins:
            
            # Check the bin in both the current day and the next day
            # to account for activities that span midnight
            for day_offset in [0, 1440]:
                # Shift the bin forward by 0 or 1440 minutes
                offset_bin_start = bin_start + day_offset
                offset_bin_end = bin_end + day_offset
                
                # Find the overlap between the activity and the current bin
                # Start of overlap = later of the two start times
                overlap_start = max(start, offset_bin_start)

                # End of overlap = earlier of the two end times
                # Add 1 to make the bin end inclusive
                overlap_end = min(end, offset_bin_end + 1)
                
                # If the overlap is positive, record this portion of the activity
                if overlap_end > overlap_start:
                    # Copy the original row so the activity information is retained
                    new_row = row.copy()

                    # Store the original time bin label using 0-1439 minute format
                    new_row["time_bin_minutes"] = f"{bin_start}-{bin_end}"

                    # Calculate how many minutes of the activity fall within this bin
                    new_row["bin_duration_minutes"] = overlap_end - overlap_start

                    # Append the expanded row to the results list
                    expanded.append(new_row)
    
                    # Stop checking the next-day version of the same bin
                    # once an overlap has already been found
                    break
    
    # Convert the expanded records into a DataFrame and reset the index
    return pd.DataFrame(expanded).reset_index(drop=True)

In [ ]:
# Apply the function above to the dataframe
df_elect_expanded = create_activities_minutes(df_elect, time_bins)
df_elect_expanded.head(3)

## 4.2 Convert time to hh:mm format

At this time, the time bins are in minutes. I would like to include the a columne where they appear in the hour-minute formating. First create a column that converts the minutes to HH:MM format.

In [ ]:
# Convert minutes to HH:MM format, handling midnight crossover
def minutes_to_hhmm(minutes):
    minutes = minutes % 1440     # Wrap around after midnight (1440 mins = 24 hours)
    hours = minutes // 60        # Get whole hours
    mins = minutes % 60          # Get remaining minutes (using the modulo)
    return f"{hours:02d}:{mins:02d}"  # Return in HH:MM format

Create anothe function that creates the column for start-end range.

In [ ]:
# Convert time_bin from minute range format to HH:MM range format
def time_bin_to_hhmm(time_bin_str):
    # Split the minute range time bin
    start, end = time_bin_str.split('-')  # split the time bin
    start = int(start)                    # the beginning of the bin is the start time for the 30 min window
    end = int(end)                        # the end of the bin is the end time for the 30 min window

    # apply the 'minutes to hhmm' function to convert the minutes to hhmm range format
    return f"{minutes_to_hhmm(start)}-{minutes_to_hhmm(end)}"

In [ ]:
# Apply the functions to the dataframe
df_elect_expanded['time_bin'] = df_elect_expanded['time_bin_minutes'].apply(time_bin_to_hhmm)
# add the startime of day in minutes
df_elect_expanded['starttime_of_day_min'] = df_elect_expanded['time_bin_minutes'].str.split('-', expand=True)[0].astype(int)
df_elect_expanded.head(3)

# 5.0 Exploratory data analysis

## 5.1 Time durations

In [ ]:
# Regroup the data
df_dur = df_elect_expanded.groupby('mainid')['duration_minutes'].sum()
df_dur = pd.DataFrame(df_dur)

In [ ]:
# Describe the percentiles
df_dur["duration_minutes"].describe(
    percentiles=[0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99])

The data is right skewed. Half of all observations fall below 495 minutes (median) while the mean is higher (≈781 minutes). Three‑quarters of all values are under about 988 minutes, but the top 5% extend beyond 2,280 minutes and the top 1% exceed 4,293 minutes, with an extreme maximum of 47,220 minutes.

In [ ]:
# Visualise the distribution of duration
plt.figure(figsize=(10,4))
sns.histplot(data=df_dur, x="duration_minutes", bins=20)
plt.xlabel("Duration (min)")
plt.title("Distribution of Duration")
plt.savefig('Distribution of Duration Spent.png')
plt.show()

This long upper tail indicates that a small minority engage in the activities for extremely long periods, and these influence the overall spread of the data. I will check the periods more than 5000 minutes

In [ ]:
# View households with more than 5000 hours of engagement
df_dur_5000more = df_dur[df_dur['duration_minutes'] > 5000]
df_dur_5000more = df_dur_5000more.reset_index()
df_dur_5000more.mainid.nunique()

## 5.2 Partcipation per Time

In [ ]:
# Plot the count of active enegements at the starttimes
df_summary = (df_elect_expanded["starttime_of_day_min"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame())

In [ ]:
# Plot the values
plt.figure(figsize=(18, 5))
sns.barplot(data=df_summary, x='starttime_of_day_min', y='count')
plt.xlabel("Time of day (minute)")
plt.ylabel("Households Engaging at Time")
plt.title("Number of Households Engaging at Time of Day")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('Number of Households Engaging at Time of Day.png')
plt.show()

While the least of household participation are observed in the mornings, the average duration of time spent on energy using activities is highest in this time. The invers is observed in the evening period

## 5.3 Average time spent within a 30 minute bin across the day

In [ ]:
# Group by the id and the time bins
df_household_time_mean = (df_elect_expanded
    .groupby(['mainid', 'time_bin'])['bin_duration_minutes']
    .mean()
    .reset_index())

In [ ]:
# Calculate the average duration at each time bin
df_household_time_mean = df_household_time_mean.groupby('time_bin').agg(
    count = ('mainid', 'nunique'), # get unique households
    average_duration = ('bin_duration_minutes', 'mean'))  # get the mean of the duration at each bin

In [ ]:
# Convert to dataframe
df_household_time_mean = pd.DataFrame(df_household_time_mean).reset_index()
df_household_time_mean

In [ ]:
# Visualise the mean durations spent by engaging housholds
plt.figure(figsize=(18, 6))

# Draw line with markers
sns.lineplot(data=df_household_time_mean, x="time_bin", y="average_duration",
             marker="o", errorbar=None)

# Add text labels on each marker
for x, row in df_household_time_mean.iterrows():
    plt.text(x=row["time_bin"], y=row["average_duration"] + 0.2,   # Offset the count on the marker vertically
             s=str(row["count"]), # show count
             ha='center', va='bottom', fontsize=9, color='black')

plt.xlabel("Time of day (hour)")      # Label the x-axis
plt.ylabel("Average Duration at Time (minutes)")   # Label y axis
plt.title("Duration per 30-min time bin")   # Add title
plt.xticks(rotation=45, ha='right') # Rotate the x-tixks
plt.tight_layout()
plt.savefig('Duration per 30-min time bin.png')
plt.show()

While more time is spent during the morning peak, there are only a few households engaging in these activities. Evenings have higher engagements.

## Save datasets

In [ ]:
# save the resampled data
df_elect.to_csv('df_elect.csv', index=False)

In [ ]:
# save the resampled data
df_elect_expanded.to_csv('resampled_tus.csv', index=False)